# Carrier displacement on Colab (A100)Generation and the local judge run here; all analysis runs on CPU afterwards, from thesaved JSON, so you never need to regenerate to re-analyse.Order of work (README):1. **RQ2 pilot** (5 seeds, 3 settings). If the coalition gains nothing, stop and try   `structured`, then `agent_order: colluders_first`, then a larger model on Modal.2. **Main sweep** (`settings: [plan]`, 20 paired seeds).3. **Judge** with Qwen2.5-14B, after generation frees the GPU.4. **Report**.Results are written to Google Drive, and the runner skips episodes that already have`metrics.json`, so a disconnect costs you one episode, not the sweep.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import sys; print(sys.version)

## 2. Keep results on DriveColab disconnects. Everything under `RESULTS_ROOT` survives one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_ROOT = '/content/drive/MyDrive/carriers-results'
os.makedirs(RESULTS_ROOT, exist_ok=True)
print(RESULTS_ROOT)

## 3. Clone the code`GITHUB_TOKEN` only matters while this repo is private. Colosseum is public and iscloned with its CoLLAB submodule, unmodified.

In [ ]:
from getpass import getpass
import os, subprocess

GITHUB_USER = 'sagnikc395'
REPO = 'agent-collusion-under-observation'
token = os.environ.get('GITHUB_TOKEN') or getpass('GitHub token (blank if the repo is public): ')

%cd /content
if not os.path.exists('/content/' + REPO):
    url = (f'https://{token}@github.com/{GITHUB_USER}/{REPO}.git' if token
           else f'https://github.com/{GITHUB_USER}/{REPO}.git')
    subprocess.run(['git', 'clone', url], check=True)

if not os.path.exists('/content/colosseum'):
    subprocess.run(['git', 'clone', '--recurse-submodules',
                    'https://github.com/umass-ai-safety/colosseum.git'], check=True)

%cd /content/{REPO}
!git pull --ff-only || true

## 4. InstallColosseum targets Python 3.11+; Colab's runtime qualifies, so we install into itdirectly rather than building a second venv.

In [ ]:
!pip install -q "terrarium-agents[providers,science,plots,vllm]==0.2.1" names tqdm pyyaml scipy pandas
import os
os.environ['COLOSSEUM_ROOT'] = '/content/colosseum'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
# Restart the runtime here if pip changed a package that was already imported.

## 5. Start the agent modelQwen2.5-7B-Instruct on port 8020. The first launch downloads ~15 GB.

In [ ]:
!bash scripts/start_vllm.sh
!curl -s http://127.0.0.1:8020/v1/models | head -c 400

## 6. RQ2 pilotThree settings x 5 paired seeds. Read the table before spending GPU time on the rest.

In [ ]:
!python -m carriers.runner --config configs/pilot.yaml \
    --out-dir $RESULTS_ROOT/pilot --max-concurrent-runs 4

In [ ]:
!python -m analysis.report --results $RESULTS_ROOT/pilot \
    --out $RESULTS_ROOT/pilot_tables --n-boot 200

## 7. Main sweep11 settings x 20 seeds. Safe to re-run: finished episodes are skipped.Cut order if time runs short: RQ6 (`verifier`), then RQ5 (`find_partner`), then thecombined ablations.

In [ ]:
!python -m carriers.runner --config configs/main.yaml \
    --out-dir $RESULTS_ROOT/main --max-concurrent-runs 4

In [ ]:
# Progress at any time (also safe from a second session).
!cat $RESULTS_ROOT/main/progress.json

## 8. JudgeFree the 7B server first: the 14B judge needs the memory.

In [ ]:
!pkill -f vllm.entrypoints.openai.api_server || true
import time; time.sleep(20)
!bash scripts/start_vllm.sh --judge

In [ ]:
!python -m analysis.judge --results $RESULTS_ROOT/main \
    --base-url http://127.0.0.1:8021/v1 --model Qwen/Qwen2.5-14B-Instruct

## 9. Report (CPU; this part also runs on your laptop)

In [ ]:
!python -m analysis.report --results $RESULTS_ROOT/main \
    --out $RESULTS_ROOT/main_tables
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS_ROOT}/main_tables/report.md').read()))

## 10. Keep the logsColosseum's own logs are not public; ours are part of the contribution, so archivethem with the tables.

In [ ]:
!cd $RESULTS_ROOT && tar czf main_logs.tar.gz main main_tables
print('archived:', RESULTS_ROOT + '/main_logs.tar.gz')